## Ноутбук с подготовкой и очисткой данных

1. Импорт библиотек и конфигурация проекта

In [2]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
}

In [3]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pyarrow

2. Загрузка и первичный осмотр данных

In [4]:
dates_only = pd.read_csv('../data/raw/raw_dataset.csv', usecols=['Дата размещения объявления'])
print("Самая поздняя дата в файле:", dates_only['Дата размещения объявления'].max())

Самая поздняя дата в файле: 2025-07-14


Заметим, что самая поздняя дата объявления - 2025-07-14. Так как файл слишком большой (5гб), отберём только объявления, размещенные с 2025-01-14 по 2025-07-14: это позволит не только облегчить вычисления, но и сделает будущую модель лучше, ведь она будет обучена на относительно "свежих" данных (учитываем инфляцию и актуальность цен).

In [5]:
date = 'Дата размещения объявления'
chunks = []
for chunk in pd.read_csv('../data/raw/raw_dataset.csv', chunksize = 100000, low_memory=False):
    filtered = chunk[chunk[date].between('2025-01-14','2025-07-14')]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index = True)
df.shape

(585855, 58)

585855 строк - оптимальное значенение для обучения модели. Больше брать смысла нет, так как качество модели растет логарифмически по отношению к объему данных. Для начала проверим, есть ли в нашем датасете информация о спецтехнике.

In [6]:
trucks_count = df['Тип техники'].notna().sum()
print(f"Найдено коммерческой техники/спецтехники: {trucks_count} шт.")

Найдено коммерческой техники/спецтехники: 0 шт.


Отлично! Никаких грузовиков, тягачей и кранов в нашем полном датасете нет - можно спокойно работать с нашим последующим сэмплом в 100к, не боясь что мы удалим важные столбцы для нелегковых автомобилей.
Все эксперименты будем проводить на DEV_MODE = True, чтобы работать с 100тыс. строк. В конце работы в CONFIG поменяем значение на False -> финальный запуск на всем объеме (585к строк)

In [7]:
if CONFIG['DEV_MODE']:
    df = df.sample(n=100000, random_state=CONFIG['RANDOM_STATE'])
    print("Режим разработки (100к строк). Всё будет летать!")
else:
    df = df.copy()
    print("Финальный режим (585к строк). Обучаем итоговую модель.")

Финальный режим (585к строк). Обучаем итоговую модель.


In [8]:
df.to_parquet('../data/raw/df_optimal.parquet') # Сохраняем сырой датасет, но с нужным количеством строк

In [9]:
df.info

<bound method DataFrame.info of              Название машины     Год  \
0       Aston Martin Vantage  2018.0   
1           Aston Martin DB9  2005.0   
2          Aston Martin DB11  2017.0   
3           Aston Martin DB9  2013.0   
4           Aston Martin DBS  2019.0   
...                      ...     ...   
585850            Volvo XC90  2006.0   
585851             Volvo C30  2008.0   
585852             Volvo V90  2019.0   
585853             Volvo V60  2018.0   
585854             Volvo S80  2002.0   

                                                   Ссылка  \
0       https://auto.drom.ru/himki/aston_martin/vantag...   
1       https://auto.drom.ru/krasnodar/aston_martin/db...   
2       https://auto.drom.ru/moscow/aston_martin/db11/...   
3       https://auto.drom.ru/moscow/aston_martin/db9/1...   
4       https://auto.drom.ru/moscow/aston_martin/dbs/5...   
...                                                   ...   
585850  https://auto.drom.ru/moscow/volvo/xc90/5051113...   

In [10]:
df.head(5)

,Название машины,Год,Ссылка,Дата размещения объявления,Цена,Кол-во просмотров,Скрыто,Объем двигателя,Тип двигателя,Мощность,...,Объем ковша,Длина стрелы,Грузоподъемность стрелы,Высота вышки,Состояние,Страна производства,Высота подъема,Ошибка_ст,Ошибка_знач,Пропуски в данных
0,Aston Martin Vantage,2018.0,https://auto.drom.ru/himki/aston_martin/vantag...,2025-04-03,11834000.0,1899.0,0.0,4.0,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"20,","открытый,","17,"
1,Aston Martin DB9,2005.0,https://auto.drom.ru/krasnodar/aston_martin/db...,2025-04-18,4999000.0,1230.0,0.0,5.9,бензин,456.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"9,","510.0,","17,"
2,Aston Martin DB11,2017.0,https://auto.drom.ru/moscow/aston_martin/db11/...,2025-05-16,13900000.0,1317.0,0.0,5.2,бензин,608.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aston Martin DB9,2013.0,https://auto.drom.ru/moscow/aston_martin/db9/1...,2025-03-29,8500000.0,14571.0,0.0,5.9,бензин,510.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aston Martin DBS,2019.0,https://auto.drom.ru/moscow/aston_martin/dbs/5...,2025-03-31,24300000.0,11888.0,0.0,5.2,бензин,715.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
(df.isnull().mean() * 100).round(2)

Название машины                 0.00
Год                             0.00
Ссылка                          0.00
Дата размещения объявления      0.00
Цена                            0.00
Кол-во просмотров               0.00
Скрыто                          0.00
Объем двигателя                 0.01
Тип двигателя                   0.00
Мощность                        0.01
Коробка передач                 0.00
Привод                          0.00
Пробег                          0.80
Руль                            0.04
Поколение                       0.01
Рестайлинг                      0.01
Цвет                            0.38
Комплектация                    0.22
Владелец                        0.00
Особые отметки                 93.42
Тип кузова                      0.40
VIN                            99.11
Проверено                     100.00
Номер кузова                   99.99
Метка                           0.00
Город                           0.00
Регион                          0.00
М

Заметим, что такие данные как высота подъема, объем ковша, высота седла, тип кабины и др. на 100% отсутствуют. Это связано с тем, что в данном датасете только легковые автомобили. Сможем смело удалять эти столбцы

3. Очистка данных

In [12]:
# Перед удалением создадим отдельный столбец, тк особые отметки - очень важный признак
df['Есть особые отметки'] = df['Особые отметки'].notna().astype(int)
# Задаем порог: если пропусков больше, чем 50% - смело удаляем столбец
threshold = len(df) * 0.5  
df_cleaned = df.dropna(thresh=threshold, axis=1).copy()
print('Было колонок: ', df.shape[1])
print('Стало колонок: ', df_cleaned.shape[1])

Было колонок:  59
Стало колонок:  27


In [13]:
# Удаляем строчки, у которых отсутствует пробег - всего 0.77 от датасета, это ни на что не повлияет
df_cleaned = df_cleaned.dropna(subset=['Пробег'])

# Избавляемся от дубликатов
df_cleaned.drop_duplicates(inplace=True)

In [14]:
# Убираем ненужны столбцы
cols_to_drop = ['Дата размещения объявления', 'Кол-во просмотров', 'Скрыто', 'Ссылка', 'Владелец', 'Пропуски в данных']
df_cleaned.drop(columns=cols_to_drop, inplace=True)

In [15]:
cols_to_unknown = ['Цвет', 'Комплектация']
for col in cols_to_unknown:
    df_cleaned[col] = df_cleaned[col].fillna('Unknown')

In [16]:
# Переименовываем "Метку" в понятную "Марку"
df_cleaned.rename(columns={'Метка': 'Марка'}, inplace=True)

4. Разделение на train и test

Критический шаг. Делаем это для избежания утечки данных

In [17]:
X = df_cleaned.drop(columns=[CONFIG['TARGET']])
y = df_cleaned[CONFIG['TARGET']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=CONFIG['RANDOM_STATE'])

5. Контекстная очистка

In [18]:
# Заполним медианой столбцы с числовыми признаками

num_cols = ['Мощность', 'Объем двигателя']
for col in num_cols:
    global_median = X_train[col].median()
    group_medians = X_train.groupby('Название машины')[col].median()
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_medians)
        ).fillna(global_median)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_medians)
    ).fillna(global_median)

In [19]:
# Заполним модой столбцы с категориальными признаками

cat_cols = ['Привод', 'Руль', 'Тип кузова', 'Тип двигателя', 'Владельцы', 'Коробка передач']
for col in cat_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby('Название машины')[col].apply(
        lambda x: x.mode().get(0, global_mode)
    )
    X_train[col] = X_train[col].fillna(
        X_train['Название машины'].map(group_modes)
    ).fillna(global_mode)
    X_test[col] = X_test[col].fillna(
        X_test['Название машины'].map(group_modes)
    ).fillna(global_mode)

In [20]:
# Особенный случай - поколение и рестайлинг зависят и от модели, и от года выпуска
special_cols = ['Поколение', 'Рестайлинг']
for col in special_cols:
    global_mode = X_train[col].mode()[0]
    group_modes = X_train.groupby(['Название машины', 'Год'])[col].apply(
        lambda x: x.mode().get(0, global_mode)
    ).rename(f'mode_{col}')

    X_train[col] = X_train[col].fillna(
        X_train.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

    X_test[col] = X_test[col].fillna(
        X_test.join(group_modes, on=['Название машины', 'Год'])[f'mode_{col}']
    ).fillna(global_mode)

In [21]:
# Проверим
display(X_test.isna().sum())
X_train.isna().sum()

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
dtype: int64

6. Проверка выбросов

In [22]:
X_train.columns

Index(['Название машины', 'Год', 'Объем двигателя', 'Тип двигателя',
       'Мощность', 'Коробка передач', 'Привод', 'Пробег', 'Руль', 'Поколение',
       'Рестайлинг', 'Цвет', 'Комплектация', 'Тип кузова', 'Марка', 'Город',
       'Регион', 'Макро-регион', 'Владельцы', 'Есть особые отметки'],
      dtype='object')

Посмотрим на год

In [23]:
X_train['Год'].value_counts().sort_index().head(30)

Год
1941.0       1
1949.0       1
1950.0       1
1953.0       1
1959.0       1
1962.0       1
1965.0       1
1967.0       1
1970.0       1
1971.0       1
1972.0       5
1973.0       8
1974.0      22
1975.0      34
1976.0      27
1977.0      45
1978.0      36
1979.0      56
1980.0      84
1981.0     123
1982.0     214
1983.0     318
1984.0     519
1985.0     590
1986.0     653
1987.0     844
1988.0    1312
1989.0    1545
1990.0    2171
1991.0    2790
Name: count, dtype: int64

Избавимся от ретро-автомобилей. Они будут ломать логику модели, ведь для обычных авто действует правило "чем старше, тем он дешевле" (цена падает с возрастом). Для ретро-машин "чем старше и раритетнее авто, тем он дороже".

In [24]:
X_train = X_train[X_train['Год'] >= 1990]
X_test = X_test[X_test['Год'] >= 1990]

Посмотрим на объем двигателя

In [25]:
X_train['Объем двигателя'].value_counts().sort_index().head(20)

Объем двигателя
0.000000     928
0.600000      62
0.700000       2
0.700000    3907
0.800000     181
0.900000       1
1.000000    6881
1.066667       1
1.071429       1
1.075000       1
1.077778       1
1.090909       1
1.100000     472
1.133333       3
1.150000       2
1.200000    6417
1.214286       1
1.214815       1
1.223077       1
1.226667       1
Name: count, dtype: int64

In [26]:
X_train.sort_values(by='Объем двигателя', ascending=True).head(10)

,Название машины,Год,Объем двигателя,Тип двигателя,Мощность,Коробка передач,Привод,Пробег,Руль,Поколение,Рестайлинг,Цвет,Комплектация,Тип кузова,Марка,Город,Регион,Макро-регион,Владельцы,Есть особые отметки
371440,Nissan Leaf,2014.0,0.0,электро,109.0,редуктор,передний,96000.0,правый,1.0,0.0,черный,G//G Aero Style//G Aero Style side/curtain air...,хэтчбек 5 дв.,nissan,Биробиджан,Еврейская автономная область,ДФО,2,0
579511,Volkswagen ID.4 Crozz,2023.0,0.0,электро,313.0,редуктор,4WD,1700.0,левый,1.0,0.0,синий,84.8 kWh Prime,джип/suv 5 дв.,volkswagen,Новосибирск,Новосибирск,Новосибирск,1,0
373120,Nissan Leaf,2016.0,0.0,электро,109.0,редуктор,передний,136000.0,правый,1.0,0.0,серый,30kWh G,хэтчбек 5 дв.,nissan,Иркутск,Иркутская область,СФО,1,0
370788,Nissan Leaf,2011.0,0.0,электро,109.0,редуктор,передний,205000.0,правый,1.0,0.0,белый,X,хэтчбек 5 дв.,nissan,Иркутск,Иркутская область,СФО,1,0
377308,Nissan Leaf,2018.0,0.0,электро,150.0,редуктор,передний,143000.0,правый,2.0,0.0,бордовый,40kWh G,хэтчбек 5 дв.,nissan,Иркутск,Иркутская область,СФО,1,0
486021,Toyota bZ4X,2022.0,0.0,электро,218.0,редуктор,4WD,17000.0,левый,1.0,0.0,серый,67 kWh X-mode Ultra/Premium,джип/suv 5 дв.,toyota,Иркутск,Иркутская область,СФО,1,0
370789,Nissan Leaf,2011.0,0.0,электро,109.0,редуктор,передний,200000.0,правый,1.0,0.0,белый,G,хэтчбек 5 дв.,nissan,Иркутск,Иркутская область,СФО,1,0
294101,Lexus RZ450e,2023.0,0.0,электро,308.0,редуктор,4WD,41000.0,левый,1.0,0.0,белый,71.4 kWh Premium,джип/suv 5 дв.,lexus,Москва,Москва,Москва,1,0
502170,Toyota bZ3,2023.0,0.0,электро,181.0,редуктор,передний,26300.0,левый,1.0,0.0,серый,50 kWh Elite PRO,седан,toyota,Владивосток,Приморский край,ДФО,1,0
327953,Mercedes-Benz EQS SUV,2022.0,0.0,электро,360.0,редуктор,4WD,16530.0,левый,1.0,0.0,черный,EQS 450 4MATIC Electric Art Advanced Plus,джип/suv 5 дв.,mercedes-benz,Краснодар,Краснодарский край,ЮФО,4 и более,0


Заметим, что все автомобили, у которых объем двигателя равен нуля - это электромобили. Ценные данные, их удалять нельзя.

In [27]:
X_train['Объем двигателя'].value_counts().sort_index(ascending=False).head(30)

Объем двигателя
15.000000      1
8.000000       1
7.400000       1
7.300000       3
7.000000       1
6.800000       6
6.700000      18
6.600000      10
6.500000       8
6.400000      28
6.300000       5
6.200000     266
6.100000       6
6.000000     105
5.900000       6
5.800000      24
5.700000     743
5.666667       1
5.600000     522
5.500000     586
5.400000      84
5.300000      97
5.250000       2
5.242857       1
5.228571       1
5.200000      60
5.090909       1
5.059211       1
5.000000     746
4.952542       2
Name: count, dtype: int64

После значения 6.8 данные резко обрываются, остаются единичные машины. Значение 0.50 - скорее всего единичный выброс. Избавимся от этого.

In [28]:
X_train = X_train[(X_train['Объем двигателя'] == 0) | ((X_train['Объем двигателя'] >= 0.6) & (X_train['Объем двигателя'] <= 6.8))]
X_test = X_test[(X_test['Объем двигателя'] == 0) | ((X_test['Объем двигателя'] >= 0.6) & (X_test['Объем двигателя'] <= 6.8))]

Посмотрим на тип двигателя.

In [29]:
X_train['Тип двигателя'].value_counts().sort_index(ascending=False).head(10)

Тип двигателя
электро          928
дизель         27325
газ/бензин         2
газ               30
бензин        430189
Name: count, dtype: int64

Газ/бензин - это реальный тип двигателя. Это не ошибка. Убирать не будем

Посмотрим на мощность.

In [30]:
X_train['Мощность'].value_counts().sort_index(ascending=False).head(10)

Мощность
1560.0    1
800.0     4
780.0     6
760.0     1
740.0     1
725.0     1
720.0     3
715.0     2
700.0     2
682.0     3
Name: count, dtype: int64

Удалим выбросы в виде значений 1952 и 1560

In [31]:
X_train = X_train[X_train['Мощность'] <= 800]
X_test = X_test[X_test['Мощность'] <= 800]

Посмотрим на простые признаки

In [32]:
# Быстрая проверка простых категориальных признаков на неявные дубликаты и аномалии
check_cols = ['Коробка передач', 'Привод', 'Тип кузова', 'Цвет', 'Рестайлинг', 'Поколение', 'Комплектация', 'Метка', 'Город', 'Макро-регион', 'Регион']

for col in check_cols:
    print(f"\tРаспределение для столбца '{col}'")
    print(df[col].value_counts())
    print("\n" + "-"*40 + "\n")

	Распределение для столбца 'Коробка передач'
Коробка передач
АКПП        250962
МКПП        230386
CVT          77491
РКПП         23886
редуктор      3108
Name: count, dtype: int64

----------------------------------------

	Распределение для столбца 'Привод'
Привод
передний                      380097
4WD                           149210
задний                         56402
двигатель посередине (MID)       120
Name: count, dtype: int64

----------------------------------------

	Распределение для столбца 'Тип кузова'
Тип кузова
седан                                                                                                                                                                   240952
джип/suv 5 дв.                                                                                                                                                          120173
хэтчбек 5 дв.                                                                                                     

Данные чистые.

Посмотрим на владельцев.

In [33]:
X_train['Владельцы'].value_counts()

Владельцы
4 и более    237492
1             97162
3             61691
2             60823
1.0             928
2.0             216
3.0             161
Name: count, dtype: int64

Заметим, что данные смешались. Попробуем привести всё к единому формату

In [34]:
owners_mapping = {
    '4 и более': 4, # Стандартное упрощение для моделей. Оно дает ей понять направление тренда (что владельцев много), не усложняя вычисления.
    '1.0': 1,
    '2.0': 2,
    '3.0': 3,
    1.0: 1,
    2.0: 2,
    3.0: 3,
    '1': 1,
    '2': 2,
    '3': 3
}

# Применяем замену к столбцу
X_train['Владельцы'] = X_train['Владельцы'].replace(owners_mapping)
X_test['Владельцы'] = X_test['Владельцы'].replace(owners_mapping)

C:\Users\Степан\AppData\Local\Temp\ipykernel_7752\764228596.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train['Владельцы'] = X_train['Владельцы'].replace(owners_mapping)
C:\Users\Степан\AppData\Local\Temp\ipykernel_7752\764228596.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test['Владельцы'] = X_test['Владельцы'].replace(owners_mapping)


Посмотрим на руль.

In [35]:
X_train['Руль'].value_counts()

Руль
левый     323150
правый    135323
Name: count, dtype: int64

In [36]:
# Удаляем этот выброс.
X_train = X_train[X_train['Руль'] != 'правый, левый']
X_test = X_test[X_test['Руль'] != 'правый, левый']

Посмотрим на пробег.

In [37]:
X_train['Пробег'].value_counts().sort_index(ascending=False).head(10)

Пробег
999999.0    274
999998.0     17
999997.0      2
999991.0      2
999990.0      6
999635.0      1
999586.0      1
999568.0      1
999555.0      2
999253.0      1
Name: count, dtype: int64

Очевидно, что почти 300 машин не могут иметь ровно 999999 км пробег. Заменим на медиану по году выпуска.

In [38]:
X_train.loc[X_train['Пробег'] >= 999000, 'Пробег'] = np.nan
X_test.loc[X_test['Пробег'] >= 999000, 'Пробег'] = np.nan
X_train['Пробег'] = X_train['Пробег'].fillna(X_train.groupby('Год')['Пробег'].transform('median'))
X_test['Пробег'] = X_test['Пробег'].fillna(X_test.groupby('Год')['Пробег'].transform('median'))

Удалим странные аномалии в Типе кузова.

In [39]:
X_train['Тип кузова'].value_counts()

Тип кузова
седан                                            187333
джип/suv 5 дв.                                    95758
хэтчбек 5 дв.                                     77889
универсал                                         36568
минивэн                                           27137
лифтбек                                           11773
джип/suv 3 дв.                                     9189
хэтчбек 3 дв.                                      5712
купе                                               2646
пикап                                              2140
седан//хэтчбек 5 дв.                                740
бортовой грузовик//минивэн                          362
открытый                                            200
бортовой грузовик                                   185
купе//открытый кузов                                160
цельнометаллический фургон                          120
открытый кузов                                       98
минивэн//хэтчбек 5 дв.               

In [42]:
X_train = X_train[~X_train['Тип кузова'].str.contains('//', na=False)]
X_test = X_test[~X_test['Тип кузова'].str.contains('//', na=False)]

In [45]:
# Проверка
X_train['Пробег'].isna().sum(), X_test['Пробег'].isna().sum()

(np.int64(0), np.int64(0))

7. Объединение и импорт

In [46]:
# Синхронизируем таргет y с очищенными признаками X по индексам
y_train = y_train.loc[X_train.index]
y_test = y_test.loc[X_test.index]

# Теперь их длины станут абсолютно одинаковыми, можно безопасно объединять
train_full = pd.concat([X_train, y_train], axis=1)
test_full = pd.concat([X_test, y_test], axis=1)

print(train_full.isna().sum(), test_full.isna().sum())


train_full.to_parquet('../data/cleaned/train_cleaned.parquet')
test_full.to_parquet('../data/cleaned/test_cleaned.parquet')

Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы              0
Есть особые отметки    0
Цена                   0
dtype: int64 Название машины        0
Год                    0
Объем двигателя        0
Тип двигателя          0
Мощность               0
Коробка передач        0
Привод                 0
Пробег                 0
Руль                   0
Поколение              0
Рестайлинг             0
Цвет                   0
Комплектация           0
Тип кузова             0
Марка                  0
Город                  0
Регион                 0
Макро-регион           0
Владельцы   